# Laplace near-field evaluation

The numerical experiment is based on the following setting:

1. A finite element surface $\Gamma_h$ obtained from a mesh of the sphere is given. The mapped FE surface itself, whether affine or curved, is regarded as the **exact geometry** of the experiment.

2. We consider

   $$
   u(\boldsymbol x)=\frac{1}{\lVert\boldsymbol x-\boldsymbol x_s\rVert},
   $$

   where $\boldsymbol x_s$ lies strictly inside the domain enclosed by $\Gamma_h$. The function is harmonic in the exterior and is represented by

   $$
   u(\boldsymbol x)=S(j)(\boldsymbol x)+D(m)(\boldsymbol x).
   $$

   Here $m=\gamma_0 u$ is the Dirichlet trace and $j=\gamma_1=-\nabla u\cdot\boldsymbol n_h$ is the Neumann trace with respect to the exterior-domain normal. The normal $\boldsymbol n_h$ used by the mesh points out of the enclosed bounded domain.

3. The traces are interpolated into the corresponding FE spaces,

   $$
   m_h=\Pi_h^1m,\qquad j_h=\Pi_h^0j.
   $$

   Their relative interpolation errors are measured on $\Gamma_h$ as in `Interpolation_Convergence_Documentation.ipynb` and reported separately. They are not directly comparable to the pointwise potential error and are therefore not drawn as error thresholds in the potential plots.

4. We compare naive Gaussian quadrature with singularity extraction and study the error as the exterior evaluation point approaches the FE surface.


## Error separation and notation

The superscript $G$ denotes naive Gaussian quadrature, whereas the star denotes singularity extraction:

$$
u_G=S^G(j_h)+D^G(m_h),\qquad u_*=S^*(j_h)+D^*(m_h).
$$

Both methods use exactly the same interpolated Cauchy data. Their total errors $E_G=|u-u_G|$ and $E_*=|u-u_*|$ therefore include the effect of FE interpolation. A high-order reference $u_{\mathrm{FE}}^{\mathrm{ref}}$ is computed using the same $m_h$ and $j_h$. We report

$$
E_{\mathrm{FE}}=|u-u_{\mathrm{FE}}^{\mathrm{ref}}|,\qquad
E_{\mathrm{quad}}^G=|u_G-u_{\mathrm{FE}}^{\mathrm{ref}}|,\qquad
E_{\mathrm{quad}}^*=|u_*-u_{\mathrm{FE}}^{\mathrm{ref}}|.
$$

The quantity $E_{\mathrm{FE}}$ is the propagated FE representation error at the target point; it must not be identified with either trace interpolation error. The relative $L^2(\Gamma_h)$ errors $\|m-m_h\|/\|m\|$ and $\|j-j_h\|/\|j\|$ are reported separately. Meanwhile, $E_{\mathrm{quad}}^G$ and $E_{\mathrm{quad}}^*$ isolate the integration algorithms. Single- and double-layer contributions are recorded separately so that cancellation cannot conceal an integration error.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import Markdown, display
import numpy as np
import pandas as pd
from netgen.occ import *
from ngsolve import *
from ngsolve.bem import *
from ngsolve.webgui import Draw

In [ ]:
FOUR_PI = 4.0 * np.pi
radius = 1.0
maxh = 0.2

curve_order = 3 # geometry 
trace_order = 3 # H1 conform elements
flux_order = trace_order - 1 # L2Surface 

gauss_orders = list(range(max(trace_order + 2, 5), 22, 2)) # numerical integration

source_point = np.array((0.20, -0.15, 0.10), dtype=float) # off-centered singularity inside the sphere

reference_bonus_intorder = 30
fmm_params = {'use_fmm': False}

if np.linalg.norm(source_point) >= radius:
    raise ValueError('source_point must lie strictly inside the sphere')

sphere = Sphere((0, 0, 0), radius)
sphere.faces.name = 'sphere'
source_mesh = Mesh(OCCGeometry(sphere).GenerateMesh(maxh=maxh)).Curve(curve_order)
source_region = source_mesh.Boundaries('sphere')

In [ ]:
desired_direction = np.array((1.0, 1.0, 1.0))
desired_direction /= np.linalg.norm(desired_direction)

# source_mesh contains volume vertices as well. Maximizing the unnormalised
# projection selects a supporting vertex on the outer boundary; comparing
# normalized directions alone could accidentally select an interior vertex.
surface_point = max(
    (np.array(vertex.point, dtype=float) for vertex in source_mesh.vertices),
    key=lambda point: np.dot(point, desired_direction),
)
approach_direction = surface_point / np.linalg.norm(surface_point)


def make_target(distance):
    """Return the Euclidean point and the MeshPoint required by the BEM potential."""
    point = surface_point + distance * approach_direction
    target_mesh_size = max(0.1 * distance, 1e-7)
    evaluation_box = Box(
        Pnt(*(point - target_mesh_size)),
        Pnt(*(point + target_mesh_size)),
    )
    target_mesh = Mesh(OCCGeometry(evaluation_box).GenerateMesh(maxh=target_mesh_size))
    return point, target_mesh, target_mesh(*point)

## Geometry and target positions

The following sketch shows the fixed FE geometry together with two explicitly constructed visualization boxes: red marks the target at the largest distance, and green marks the target at the smallest distance. Their common box size is intentionally exaggerated so that both positions remain visible; these boxes are not used by the quadrature experiment.


In [ ]:
evaluation_distances = [10**(-i) for i in range(1, 7)]
max_distance = max(evaluation_distances)
min_distance = min(evaluation_distances)

# These two boxes are visualization objects only. They are deliberately
# constructed here, independently of the technical mesh in make_target.
max_target_point = surface_point + max_distance * approach_direction
min_target_point = surface_point + min_distance * approach_direction
visualization_box_halfwidth = 0.025
max_target_box = Box(
    Pnt(*(max_target_point - visualization_box_halfwidth)),
    Pnt(*(max_target_point + visualization_box_halfwidth)),
)
min_target_box = Box(
    Pnt(*(min_target_point - visualization_box_halfwidth)),
    Pnt(*(min_target_point + visualization_box_halfwidth)),
)
max_target_box.faces.name = 'target-max-distance'
min_target_box.faces.name = 'target-min-distance'
max_target_box.faces.col = (1.0, 0.2, 0.2)
min_target_box.faces.col = (0.2, 0.7, 0.2)

distance_to_source = sqrt(
    (x - source_point[0]) ** 2
    + (y - source_point[1]) ** 2
    + (z - source_point[2]) ** 2
)
u_exact = 1.0 / distance_to_source
grad_u_exact = CF((u_exact.Diff(x), u_exact.Diff(y), u_exact.Diff(z)))
normal_h = specialcf.normal(3)
m_exact = u_exact
# Exterior Neumann trace: the exterior-domain normal is -normal_h.
j_exact = -(grad_u_exact * normal_h)
print(f'approached point on Gamma_h: {tuple(surface_point)}')
print(f'surface-point radius: {np.linalg.norm(surface_point):.12e}')
print(f'max-target radius: {np.linalg.norm(max_target_point):.12e}')
print(f'min-target radius: {np.linalg.norm(min_target_point):.12e}')
print(f'maximum approach distance: {max_distance:.1e}')
print(f'minimum approach distance: {min_distance:.1e}')
print(f'evaluation distances: {evaluation_distances}')

In [ ]:
visualization_geometry = Compound([sphere, max_target_box, min_target_box])
visualization_mesh = Mesh(
    OCCGeometry(visualization_geometry).GenerateMesh(maxh=maxh)
).Curve(curve_order)
Draw(visualization_mesh)

In [ ]:
def interpolate_cauchy_data(trace_order, flux_order):
    trace_space = H1(source_mesh, order=trace_order, definedon=source_region)
    flux_space = SurfaceL2(source_mesh, order=flux_order, dual_mapping=False, definedon=source_region)
    m_h = GridFunction(trace_space, name='m_h')
    j_h = GridFunction(flux_space, name='j_h')
    with TaskManager():
        m_h.Interpolate(m_exact, definedon=source_region)
        j_h.Interpolate(j_exact, definedon=source_region)
        intorder = 2 * max(trace_order, flux_order) + 12
        m_error = sqrt(Integrate((m_h-m_exact)**2, source_mesh, definedon=source_region, order=intorder) / Integrate(m_exact**2, source_mesh, definedon=source_region, order=intorder))
        j_error = sqrt(Integrate((j_h-j_exact)**2, source_mesh, definedon=source_region, order=intorder) / Integrate(j_exact**2, source_mesh, definedon=source_region, order=intorder))
    return trace_space, m_h, flux_space, j_h, float(m_error), float(j_error)


def evaluate_star(trace_space, m_h, flux_space, j_h, bonus_intorder, target_point):
    m_trial = trace_space.TrialFunction()
    j_trial = flux_space.TrialFunction()
    with TaskManager():
        sl = LaplaceSL(j_trial * ds(bonus_intorder=bonus_intorder), **fmm_params)(j_h)
        dl = LaplaceDL(m_trial * ds(bonus_intorder=bonus_intorder), **fmm_params)(m_h)
        return float(sl(target_point)), float(dl(target_point))


def evaluate_naive_gauss(m_h, j_h, gauss_order, target_point):
    tx, ty, tz = target_point
    dx, dy, dz = tx-x, ty-y, tz-z
    distance = sqrt(dx**2 + dy**2 + dz**2)
    sl_integrand = j_h / (FOUR_PI * distance)
    dl_kernel = (normal_h[0]*dx + normal_h[1]*dy + normal_h[2]*dz) / (FOUR_PI * distance**3)
    with TaskManager():
        sl = Integrate(sl_integrand, source_mesh, definedon=source_region, order=gauss_order)
        dl = Integrate(dl_kernel*m_h, source_mesh, definedon=source_region, order=gauss_order)
        return float(sl), float(dl)


def relative_error(value, reference):
    # return abs(value - reference) / abs(reference)
    return abs(value - reference) / abs(reference)

## Experiment: naive Gauss versus singularity extraction

### Experiment profile

The following table is generated from the current parameter values, so it is updated automatically when the setup cell is rerun.

**Expectation.** For a fixed Gaussian order, naive Gauss quadrature is expected to deteriorate as $\delta\to0$, because the kernels become increasingly peaked on the nearby source element. Increasing the Gaussian order should postpone, but not remove, this loss of accuracy. Singularity extraction should remain stable and converge to the propagated FE representation error $E_{\mathrm{FE}}$; approaching $\Gamma_h$ should therefore not introduce an additional loss of attainable accuracy.


In [ ]:
display(Markdown(rf"""
| Quantity | Treatment in this experiment |
|---|---|
| FE geometry $\Gamma_h$ | fixed within one run: `maxh = {maxh}` |
| Geometry order | fixed within one run: `curve_order = {curve_order}` |
| Pole $\boldsymbol x_s$ | fixed at `{tuple(float(value) for value in source_point)}` |
| Trace spaces | fixed within one run: `H1(order={trace_order})`, `SurfaceL2(order={flux_order})` |
| Cauchy data | fixed interpolants $m_h=\Pi_h^0m$ and $j_h=\Pi_h^1j$ |
| Quadrature method | varied: naive Gauss ($G$) versus singularity extraction ($*$) |
| Quadrature orders | varied: `{gauss_orders}` |
| Approach distances | varied: `{evaluation_distances}` |
| Reference order | `reference_bonus_intorder = {reference_bonus_intorder}` |
"""))

In [ ]:
trace_space, m_h, flux_space, j_h, m_h_error, j_h_error = interpolate_cauchy_data(trace_order, flux_order)
print(f'H1 trace: p={trace_order}, ndof={trace_space.ndof}, relative interpolation error={m_h_error:.3e}')
print(f'SurfaceL2 flux: p={flux_order}, ndof={flux_space.ndof}, relative interpolation error={j_h_error:.3e}')

In [ ]:
quadrature_rows = []
for distance in evaluation_distances:
    point, target_mesh, target_point = make_target(distance)
    reference = 1.0 / np.linalg.norm(point - source_point)
    ref_sl, ref_dl = evaluate_star(trace_space, m_h, flux_space, j_h, reference_bonus_intorder, target_point)
    reference_fe = ref_sl + ref_dl
    fe_representation_error = relative_error(reference_fe, reference)
    print(f'\ndelta={distance:.0e}')

    for gauss_order in gauss_orders:
        bonus = gauss_order
        star_sl, star_dl = evaluate_star(trace_space, m_h, flux_space, j_h, bonus, target_point)
        gauss_sl, gauss_dl = evaluate_naive_gauss(m_h, j_h, gauss_order, point)

        for method, sl_value, dl_value in (
            ('singularity extraction (*)', star_sl, star_dl),
            ('naive Gauss (G)', gauss_sl, gauss_dl),
        ):
            value = sl_value + dl_value
            quadrature_rows.append({
                'method': method, 'distance': distance, 'gauss_order': gauss_order,
                'exact_value': reference, 'reference_fe': reference_fe, 'value': value,
                'total_relative_error': relative_error(value, reference),
                'quadrature_relative_error': relative_error(value, reference_fe),
                'fe_representation_error': fe_representation_error,
                'sl_quadrature_error': abs(sl_value-ref_sl)/abs(reference_fe),
                'dl_quadrature_error': abs(dl_value-ref_dl)/abs(reference_fe),
            })
        print(
            f'q={gauss_order:2d}: E_FE={fe_representation_error:.3e}, '
            f'E_*={relative_error(star_sl+star_dl, reference):.3e}, '
            f'E_G={relative_error(gauss_sl+gauss_dl, reference):.3e}'
        )

#quadrature_results = pd.DataFrame(quadrature_rows)
#quadrature_results

In [ ]:
colors = plt.cm.viridis(np.linspace(0.08, 0.92, len(evaluation_distances)))


def plot_total_error(method, method_symbol, title, filename):
    fig, ax = plt.subplots(figsize=(8.8, 4.8))

    for distance, color in zip(evaluation_distances, colors):
        data = quadrature_results[
            (quadrature_results['method'] == method)
            & (quadrature_results['distance'] == distance)
        ]
        ax.semilogy(
            data['gauss_order'],
            data['total_relative_error'],
            color=color, marker='o', linestyle='-', linewidth=1.5,
        )
        ax.axhline(
            data['fe_representation_error'].iloc[0],
            color=color, linestyle=':', linewidth=1.25,
        )

    distance_legend = ax.legend(
        handles=[
            Line2D([0], [0], color=color, linewidth=2, label=rf'$\delta={distance:.0e}$')
            for distance, color in zip(evaluation_distances, colors)
        ],
        title='approach distance', loc='upper right',
    )
    ax.add_artist(distance_legend)
    ax.legend(
        handles=[
            Line2D([0], [0], color='black', marker='o', linestyle='-', label=rf'$E_{{{method_symbol}}}$'),
            Line2D([0], [0], color='black', linestyle=':', label=r'$E_{\mathrm{FE}}$: propagated FE error'),
        ],
        title='error quantity', loc='upper left',
    )
    ax.set_xlabel('quadrature order')
    ax.set_ylabel('relative error against analytical $u$')
    ax.set_title(title)
    ax.grid(True, which='both', alpha=0.3)
    fig.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()


plot_total_error(
    method='naive Gauss (G)',
    method_symbol='G',
    title=f'Naive Gaussian quadrature | geometry order={curve_order}, H1={trace_order}, SurfaceL2={flux_order}',
    filename=f'nearfield_gauss_maxh={maxh}_H1={trace_order}_L2={flux_order}.png',
)

plot_total_error(
    method='singularity extraction (*)',
    method_symbol='*',
    title=f'Singularity extraction | geometry order={curve_order}, H1={trace_order}, SurfaceL2={flux_order}',
    filename=f'nearfield_singularity_extraction_maxh={maxh}_H1={trace_order}_L2={flux_order}.png',
)

## Direct comparison at the largest and smallest distance

The following plot compares naive Gaussian quadrature and singularity extraction directly for $\delta=10^{-1}$ and $\delta=10^{-6}$. Only the relative total error against the analytical exterior solution is shown; no numerical reference solution or additional reference line enters this comparison.


In [ ]:
comparison_distances = [1e-1, 1e-6]
comparison_styles = {
    'naive Gauss (G)': {'marker': 's', 'linestyle': '--', 'color': 'tab:orange', 'label': 'naive Gauss'},
    'singularity extraction (*)': {'marker': 'o', 'linestyle': '-', 'color': 'tab:blue', 'label': 'singularity extraction'},
}

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharey=True)
for ax, distance in zip(axes, comparison_distances):
    for method, style in comparison_styles.items():
        data = quadrature_results[
            (quadrature_results['method'] == method)
            & np.isclose(quadrature_results['distance'], distance)
        ]
        ax.semilogy(
            data['gauss_order'],
            data['total_relative_error'],
            marker=style['marker'],
            linestyle=style['linestyle'],
            color=style['color'],
            linewidth=1.7,
            label=style['label'],
        )
    ax.set_xlabel('quadrature order')
    ax.set_title(rf'$\delta={distance:.0e}$')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(loc='best')

axes[0].set_ylabel('relative total error against analytical $u$')
fig.suptitle(
    f'Direct quadrature comparison | geometry order={curve_order}, '
    f'H1={trace_order}, SurfaceL2={flux_order}'
)
fig.tight_layout()
plt.savefig(
    f'nearfield_direct_comparison_curve={curve_order}_H1={trace_order}_L2={flux_order}.png',
    dpi=300,
)
plt.show()

## Limitations

The experiment shows a qualitative difference between the two numerical integration methods. As the target approaches $\Gamma_h$, naive Gaussian quadrature loses accuracy, whereas singularity extraction remains much closer to the accuracy attainable with the chosen FE traces. This supports the claim that singularity extraction handles the near-field character of the **direct ansatz** substantially better.

This observation must not be interpreted as a complete accuracy statement for the individual layer potentials. The approximation considered here is always the sum of single and double layer potential.

Consequently, errors in the single- and double-layer contributions may reinforce or cancel each other. Moreover, the single-layer kernel is less singular than the double-layer kernel. The single-layer evaluation may therefore be considerably more accurate than the double-layer evaluation without this being visible in the error of their sum. The present experiment alone cannot establish separate convergence or accuracy properties for $S$ and $D$; dedicated tests of the individual potentials and their trace or jump relations are required for such statements.

A second limitation is the interpolation of the Cauchy data. The comparison of $u_{\mathrm{app}}$ with the analytical exterior solution necessarily contains the effects of replacing $m$ and $j$ by $m_h$ and $j_h$. A high-order evaluation with the same FE densities can isolate the quadrature error by comparison, but it cannot remove the interpolation contribution from the total error against $u$. Hence, once the quadrature error becomes sufficiently small, the present curves describe the propagated FE representation error rather than the quadrature method alone.